# FOGGIE Simulation Analysis

## Notebook Structure

- Setup & Imports
- Load Data
- Exploratory Visualisation
- 1D Spherical Analysis
- 2D Cylindrical Analysis
- Model Comparison
- Results & Interpretation

In [ ]:
import sys
print(sys.executable)
import socket
print(socket.gethostname())
import os
os.chdir("/home/shanson/foggie-dynamics")
print(os.getcwd())

In [ ]:
import yt # type: ignore
yt.set_log_level("warning")
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm

%matplotlib inline

import src
from src import (
    ProjectPaths,
    DensityProfile,
    load_hdf5,
    load_config,
)
    
PATHS = ProjectPaths.create(
    code_root=Path("/home/shanson/foggie-dynamics"),
    data_root=Path("/cephfs/shanson/FOGGIE")
)
RESULTS_DIR = PATHS.results
DATA_DIR = PATHS.data
PLOTS_DIR = PATHS.plots

config = load_config(str(PATHS.config_file()))

halos = ["halo_002392", "halo_002878", "halo_004123", "halo_005016", "halo_005036", "halo_008508"]
names = ['Hurricane', 'Cyclone', 'Blizzard', 'Squall', 'Maelstrom', 'Tempest']


## Useful Plots

In [ ]:
data_path = PATHS.snapshot_data("halo_008508", 0.0)

DMProfile = DensityProfile.from_hdf5(f"{data_path}/3d_profile_dm.h5", "dm", False)
StarProfile = DensityProfile.from_hdf5(f"{data_path}/3d_profile_star.h5", "star", False)
GasProfile = DensityProfile.from_hdf5(f"{data_path}/3d_profile_gas.h5", "gas", False)

M_dm = np.cumsum(4*np.pi * DMProfile.r_1d**2 * DMProfile.rho_1d * np.gradient(DMProfile.r_1d))
M_star = np.cumsum(4*np.pi * StarProfile.r_1d**2 * StarProfile.rho_1d * np.gradient(StarProfile.r_1d))
M_gas = np.cumsum(4*np.pi * GasProfile.r_1d**2 * GasProfile.rho_1d * np.gradient(GasProfile.r_1d))

print(f"DM to baryon ratio at R200 = {M_dm[-1] / (M_star[-1] + M_gas[-1]):.3f}")

plt.figure(figsize=(10, 6))
plt.plot(DMProfile.r_1d, M_dm, label='DM', color='orange')
plt.plot(StarProfile.r_1d, M_star, label='Stars', color='blue')
plt.plot(GasProfile.r_1d, M_gas, label='Gas', color='green')
plt.plot(DMProfile.r_1d, M_dm + M_star + M_gas, 'k-', lw=2, label='Total')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('r (kpc)')
plt.ylabel('M(<r) [Msun]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.title('Cumulative Mass Profile')

plt.show()

In [ ]:
# Build path explorer
def get_all_paths(d, prefix=''):
    """Recursively get all paths in nested dict."""
    paths = []
    for key, value in d.items():
        current_path = f"{prefix}/{key}" if prefix else key
        if isinstance(value, dict):
            paths.extend(get_all_paths(value, current_path))
        else:
            paths.append(current_path)
    return paths

def get_value_from_path(d, path):
    """Navigate nested dict using path string."""
    keys = path.split('/')
    value = d
    for key in keys:
        value = value[key]
    return value

# Get all paths
halo = "halo_005036"
results = load_hdf5(f"{RESULTS_DIR}/{halo}/evolution_time_series.h5")
all_paths = get_all_paths(results)
print(f"Found {len(all_paths)} data paths")
print("\nFirst 10 paths:")
for p in all_paths[:10]:
    print(f"  {p}")

# Simple function to view a specific path
def view(path, plot=True, context=3):
    """View data at a specific path within results dict."""
    print(f"\n{'='*80}")
    print(f"Path: {path}")
    print('='*80)
    
    # Handle invalid paths gracefully
    try:
        value = get_value_from_path(results, path)
    except KeyError as e:
        print(f"Invalid path! Key not found: {e}")
        # Suggest similar paths
        parent = '/'.join(path.split('/')[:-1])
        try:
            parent_val = get_value_from_path(results, parent) if parent else results
            if isinstance(parent_val, dict):
                print(f"Available keys at '{parent or 'root'}': {list(parent_val.keys())}")
        except KeyError:
            pass
        return None
    
    # Display based on type
    if isinstance(value, (str, np.str_)):
        print(f"Value: {value}")
    
    elif isinstance(value, (int, float, np.number)):
        # Show both scientific and regular format for readability
        print(f"Value: {value:.6e}  ({value:,.6g})")
    
    elif isinstance(value, np.ndarray):
        print(f"Shape: {value.shape}, Dtype: {value.dtype}")
        
        if np.issubdtype(value.dtype, np.number):
            # Check for NaN/Inf
            n_nan = np.sum(np.isnan(value)) if np.issubdtype(value.dtype, np.floating) else 0
            n_inf = np.sum(np.isinf(value)) if np.issubdtype(value.dtype, np.floating) else 0
            
            valid = value[np.isfinite(value)] if np.issubdtype(value.dtype, np.floating) else value
            if valid.size > 0:
                print(f"Min: {np.min(valid):.3e}, Max: {np.max(valid):.3e}, "
                      f"Mean: {np.mean(valid):.3e}, Std: {np.std(valid):.3e}")
            if n_nan or n_inf:
                print(f"Warning: {n_nan} NaN, {n_inf} Inf values")
        
        # Show data preview
        if value.size <= 2 * context:
            print(f"Data: {value}")
        else:
            print(f"First {context}: {value.flat[:context]}")
            print(f"Last {context}:  {value.flat[-context:]}")
        
        # Plot if 1D numeric
        if plot and value.ndim == 1 and value.size > 1 and np.issubdtype(value.dtype, np.number):
            fig, ax = plt.subplots(figsize=(10, 3))
            ax.plot(value, 'o-', markersize=4)
            ax.set_title(path.split('/')[-1])  # Shorter title
            ax.set_xlabel('Index')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
    else:
        print(f"Type: {type(value).__name__}")
        print(f"Value: {value}")
    
    return value

# Example: view the first path
view(all_paths[0])

In [ ]:
# Interactive widget version (run this cell to get a dropdown)
from ipywidgets import interact

@interact(path=all_paths)
def interactive_view(path):
    redshifts = view(path)

In [ ]:
from src import calc_masses_with_uncertainties, FittingResult

halo = halos[-1]
name = names[-1]
redshift = 0.0

results_path = PATHS.snapshot_results(halo, redshift)
results = {'star': {}, 'gas': {}, 'dm': {}}
for comp in ['star', 'gas', 'dm']:
    for reg in ['bulge', 'disk', 'halo']:
        if (results_path/f"{comp}_{reg}_results.h5").exists():
            results[comp][reg] = FittingResult.from_hdf5(results_path/f"{comp}_{reg}_results.h5")
        
radii = np.array([5, 10, 15, 20, 30, 50, 75, 100, 150, 200])

masses = calc_masses_with_uncertainties(results['dm'], radii)

print("DM halo masses")
for i, r in enumerate(radii):
    print(f"r={r} kpc, M(<r) = {masses['halo'][i]:.2e} + {masses[f'halo_hi'][i]:.2e} - {masses[f'halo_lo'][i]:.2e} M_sun")


In [ ]:
fig, ax = plt.subplots(figsize=[8, 6])
ax.errorbar(radii, masses['halo'], yerr=[masses['halo_lo'], masses['halo_hi']],
             fmt='none', ecolor='black', capsize=2)  # errors only, no markers
ax.plot(radii, masses['halo'], 'o', c='red', zorder=5)  # add markers separately
ax.set_xlabel('Radii [kpc]')
ax.set_ylabel(r"M(<r) $[M_\odot]$")
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_title(f'{halo}/{name}Dark Matter Halo mass')
plt.show()

In [ ]:
from typing import Dict, Optional
from src import FittingResult, get_component_colors, get_plot_config, save_figure

def plot_halo_shape_evolution(
    halo: str,
    name: str,
    config: Dict,
    PATHS: ProjectPaths,
    save_fig: bool = False,
    show_fig: bool = False,
    experiment: Optional[str] = None,
):
    """
    Plot the evolution of two-power density profile parameters (alpha, beta, rs)
    across redshift for all components (star, gas, dm).

    Parameters
    ----------
    halo : str
        Halo identifier (e.g., 'halo_008508')
    name : str
        Human-readable halo name (e.g., 'Tempest')
    config : Dict
        Configuration dictionary
    PATHS : ProjectPaths
        Project paths object
    save_fig : bool, optional
        Whether to save the figure
    """
    from src.workflow import FittingResult

    plt_cfg = get_plot_config(config, 'halo_shape')
    components = ["star", "gas", "dm"]
    params = ['alpha', 'beta', 'rs']

    # Load evolution data to get redshifts
    if experiment is not None:
        halo_ev_path = PATHS.results / experiment / halo / "evolution_time_series.h5"
    else:
        halo_ev_path = PATHS.results / halo / "evolution_time_series.h5"
    halo_evolution_data = load_hdf5(halo_ev_path)
    redshifts = halo_evolution_data['time']['redshifts']

    # Initialize results structure
    results: Dict[str, Dict] = {comp: {} for comp in components}
    for comp in components:
        for param in params:
            results[comp][param] = []
            results[comp][f'{param}_high'] = []
            results[comp][f'{param}_low'] = []
        results[comp]['redshifts_plt'] = []

    # Collect data for each component and redshift
    for comp in components:
        for redshift in redshifts:
            plot_redshift = np.log(1 + redshift)
            results_path = PATHS.snapshot_results(halo, redshift, experiment)
            results_file = results_path / f"{comp}_halo_results.h5"

            if not results_file.exists():
                continue

            halo_res = FittingResult.from_hdf5(results_file)
            summary = halo_res.summary

            if summary is None:
                continue

            # Handle NFW model (fixed alpha=1, beta=3)
            if halo_res.best_model == "NFW":
                results[comp]['alpha'].append(1)
                results[comp]['beta'].append(3)
                results[comp]['alpha_high'].append(1)
                results[comp]['alpha_low'].append(1)
                results[comp]['beta_high'].append(3)
                results[comp]['beta_low'].append(3)
                rs = summary['rs']
                rs_err = summary['rs_err']
                results[comp]['rs'].append(rs)
                results[comp]['rs_low'].append(rs_err[0])
                results[comp]['rs_high'].append(rs_err[1])
            else:
                for param in params:
                    results[comp][param].append(summary[param])
                    err_bounds = summary[f"{param}_err"]
                    results[comp][f'{param}_low'].append(err_bounds[0])
                    results[comp][f'{param}_high'].append(err_bounds[1])

            results[comp]['redshifts_plt'].append(plot_redshift)

    # Create the plot
    fig, axes = plt.subplots(
        3, 1,
        figsize=plt_cfg.get('figsize', (8, 12)),
        sharex=True
    )
    comp_colors = get_component_colors(config, 'component_type')

    param_labels = {
        'alpha': r'$\alpha$ (inner slope)',
        'beta': r'$\beta$ (outer slope)',
        'rs': r'$r_s$ [kpc]'
    }

    for ax, param in zip(axes, params):
        ax.set_ylabel(param_labels.get(param, param))

        for comp in components:
            x = np.array(results[comp]['redshifts_plt'])
            y = np.array(results[comp][param])
            y_low = np.array(results[comp][f'{param}_low'])
            y_high = np.array(results[comp][f'{param}_high'])

            if len(x) == 0:
                continue

            # Sort by x before plotting
            sort_idx = np.argsort(x)
            x = x[sort_idx]
            y = y[sort_idx]
            y_low = y_low[sort_idx]
            y_high = y_high[sort_idx]

            color = comp_colors.get(comp, 'gray')
            ax.plot(x, y, '-o', label=comp, color=color,
                   markersize=plt_cfg.get('markersize', 5))
            ax.fill_between(x, y_low, y_high, color=color, alpha=0.2)
            ax.axhline(
                np.mean(y), color=color, linestyle='--',
                linewidth=2, alpha=0.7, label=f'Mean {param}'
            )

        if param == 'rs':
            ax.set_yscale('log')
        if param == 'alpha':
            ax.axhline(1, color='black', linestyle='--', linewidth=1.5,
                      label='NFW', alpha=0.8)
        if param == 'beta':
            ax.axhline(3, color='black', linestyle='--', linewidth=1.5,
                      label='NFW', alpha=0.8)

    # Add linear redshift scale on top axis
    def ln1pz_to_z(ln1pz):
        """Convert ln(1+z) to z"""
        return np.exp(ln1pz) - 1

    def z_to_ln1pz(z):
        """Convert z to ln(1+z)"""
        return np.log(1 + z)

    for i, ax in enumerate(axes):
        # create secondary axis
        secax = ax.secondary_xaxis('top', functions=(ln1pz_to_z, z_to_ln1pz))
        
        # y-axis ticks
        ax.tick_params(axis='y', which='both', left=True, right=False,
                       labelleft=True, direction='in')
        # x-axis
        if i == 0:
            ax.legend(loc='best')
            secax.set_xlabel(r'Redshift $z$')
            ax.tick_params(axis='x', which='both', bottom=True, top=False,
                      labelbottom=False, direction='in')
            secax.tick_params(axis='x', which='both', bottom=False, top=True,
                              labeltop=True, direction='in')
        elif i == len(axes) - 1:
            ax.set_xlabel(r'$\ln(1+z)$')
            ax.invert_xaxis()
            ax.tick_params(axis='x', which='both', bottom=True, top=False,
                      labelbottom=True, direction='in')
            secax.tick_params(axis='x', which='both', bottom=False, top=True,
                              labeltop=False, direction='in')
        else:
            ax.tick_params(axis='x', which='both', bottom=True, top=False,
                      labelbottom=False)
            secax.tick_params(axis='x', which='both', bottom=False, top=True,
                              labeltop=False, direction='in')

    fig.suptitle(f"{halo} / {name}\nHalo Shape Parameter Evolution")
    plt.tight_layout()

    if save_fig:
        save_figure(
            fig,
            (f"{PATHS.plots}/{experiment}/{halo}/halo_shape_evolution.png")
            if experiment else f"{PATHS.plots}/{halo}/halo_shape_evolution.png", 
            plt_cfg.get('dpi', 200)
        )
    if show_fig:
        return fig
    else:
        plt.close(fig)

fig = plot_halo_shape_evolution(
    halos[4], names[4],
    config, PATHS,
    save_fig=False,
    show_fig=True,
    experiment='r200_scaling'
)

plt.show()

In [ ]:
# plot dm halo for redshifts
from src import (
    FittingResult,
    get_plot_config,
    ModelRegistry,
)

halo = halos[3]
name = names[3]

# Load evolution data to get redshifts
halo_ev_path = PATHS.results / halo / "evolution_time_series.h5"
halo_evolution_data = load_hdf5(halo_ev_path)
redshifts = halo_evolution_data['time']['redshifts']

plt_cfg = get_plot_config(config, 'profile_1d')
fig, ax = plt.subplots(figsize=plt_cfg.get('figsize', [10, 6]))

redshifts_plot = []
for redshift in redshifts:
    results_path = PATHS.snapshot_results(halo, redshift)/f"dm_halo_results.h5"
    if results_path.exists():
        redshifts_plot.append(redshift)

redshifts_plot = np.array(redshifts_plot)
log1pz = np.log10(1+redshifts_plot)

norm = mcolors.Normalize(vmin=log1pz.min(), vmax=log1pz.max())
cmap = plt.get_cmap(plt_cfg.get('cmap', 'turbo_r'))
sm = cm.ScalarMappable(cmap=cmap, norm=norm)

radii = np.logspace(-1, 2.5, 100)

# redshift 0 profile
results_path = PATHS.snapshot_results(halo, 0)/f"dm_halo_results.h5"
results = FittingResult.from_hdf5(results_path)

summary = results.summary
    
evaluator = ModelRegistry.get_evaluator(results.best_model)
if summary is not None:
    profile_z_0 = evaluator((radii,), summary)

for z in redshifts_plot:  
    results_path = PATHS.snapshot_results(halo, z)/f"dm_halo_results.h5"
    results = FittingResult.from_hdf5(results_path)

    summary = results.summary
    if summary is None:
        continue
        
    evaluator = ModelRegistry.get_evaluator(results.best_model)
    model_arr = evaluator((radii,), summary)
    
    # plot normalised profile divide model_arr by profile_z_0
    plot_arr = np.divide(
        model_arr, profile_z_0
    )
    #print(f"r=1 kpc, z={z:.3f}, normalised rho = {plot_arr[np.argmin(np.abs(radii - 1))]:.3f}")
    
    ax.plot(
        radii, plot_arr, '-', color=cmap(norm(np.log10(1 + z))),
        markersize=plt_cfg.get('markersize', 4),
        linewidth=plt_cfg.get('linewidth', 1)
    )
ax.set_title(f'{halo} - {name} - DM Halo evolution')

cbar = fig.colorbar(sm, ax=ax)
cbar.set_label(r'$\log_{10}(1+z)$')

ax.set_ylabel(r"$\rho$(r, z) / $\rho$(r, 0)")
ax.set_xlabel("Radii [kpc]")
ax.set_xscale('log')
plt.show()

In [ ]:
def plt_z_to_z(log_1pz: float) -> float:
    return 10**log_1pz - 1

def z_to_plt_z(z: float) -> float:
    if z >= 0:
        return np.log(1 + z)
    else:
        raise RuntimeError('cant have redshift less than 0 thicko')

print(plt_z_to_z(0.25))
print(z_to_plt_z(99))

In [ ]:
n = 100
arr = np.zeros(n)
redshifts = np.linspace(0.1, 10, n)
for i, z in enumerate(redshifts):
    arr[i] = z_to_plt_z(z)
plt.plot(redshifts, arr)
plt.xlabel("Redshift (z)")
plt.ylabel("log_10(1+z)")

plt.show()

# DM only run


In [ ]:
from typing import Dict, Optional
from src import get_plot_config, get_component_colors, save_figure


def plot_halo_shape_evolution(
    halo: str,
    name: str,
    config: Dict,
    PATHS: ProjectPaths,
    save_fig: bool = False,
    show_fig: bool = False,
    experiment: Optional[str] = None,
):
    """
    Plot the evolution of two-power density profile parameters (alpha, beta, rs)
    across redshift for all components (star, gas, dm).

    Parameters
    ----------
    halo : str
        Halo identifier (e.g., 'halo_008508')
    name : str
        Human-readable halo name (e.g., 'Tempest')
    config : Dict
        Configuration dictionary
    PATHS : ProjectPaths
        Project paths object
    save_fig : bool, optional
        Whether to save the figure
    """
    from src.workflow import FittingResult

    plt_cfg = get_plot_config(config, 'halo_shape')
    components = ["star", "gas", "dm"]
    params = ['alpha', 'beta', 'rs']

    # Load evolution data to get redshifts
    if experiment is not None:
        halo_ev_path = PATHS.results / experiment / halo / "evolution_time_series.h5"
    else:
        halo_ev_path = PATHS.results / halo / "evolution_time_series.h5"
    halo_evolution_data = load_hdf5(halo_ev_path)
    redshifts = halo_evolution_data['time']['redshifts']

    # Initialize results structure
    results: Dict[str, Dict] = {comp: {} for comp in components}
    for comp in components:
        for param in params:
            results[comp][param] = []
            results[comp][f'{param}_high'] = []
            results[comp][f'{param}_low'] = []
        results[comp]['redshifts_plt'] = []

    # Collect data for each component and redshift
    for comp in components:
        for redshift in redshifts:
            plot_redshift = np.log(1 + redshift)
            results_path = PATHS.snapshot_results(halo, redshift, experiment)
            results_file = results_path / f"{comp}_halo_results.h5"

            if not results_file.exists():
                continue

            halo_res = FittingResult.from_hdf5(results_file)
            summary = halo_res.summary

            if summary is None:
                continue

            # Handle NFW model (fixed alpha=1, beta=3)
            if halo_res.best_model == "NFW":
                results[comp]['alpha'].append(1)
                results[comp]['beta'].append(3)
                results[comp]['alpha_high'].append(1)
                results[comp]['alpha_low'].append(1)
                results[comp]['beta_high'].append(3)
                results[comp]['beta_low'].append(3)
                rs = summary['rs']
                rs_err = summary['rs_err']
                results[comp]['rs'].append(rs)
                results[comp]['rs_low'].append(rs_err[0])
                results[comp]['rs_high'].append(rs_err[1])
            else:
                for param in params:
                    results[comp][param].append(summary[param])
                    err_bounds = summary[f"{param}_err"]
                    results[comp][f'{param}_low'].append(err_bounds[0])
                    results[comp][f'{param}_high'].append(err_bounds[1])

            results[comp]['redshifts_plt'].append(plot_redshift)

    # Create the plot
    fig, axes = plt.subplots(
        3, 1,
        figsize=plt_cfg.get('figsize', (8, 12)),
        sharex=True
    )
    comp_colors = get_component_colors(config, 'component_type')

    param_labels = {
        'alpha': r'$\alpha$ (inner slope)',
        'beta': r'$\beta$ (outer slope)',
        'rs': r'$r_s$ [kpc]'
    }

    for ax, param in zip(axes, params):
        ax.set_ylabel(param_labels.get(param, param))

        for comp in components:
            x = np.array(results[comp]['redshifts_plt'])
            y = np.array(results[comp][param])
            y_low = np.array(results[comp][f'{param}_low'])
            y_high = np.array(results[comp][f'{param}_high'])

            if len(x) == 0:
                continue

            # Sort by x before plotting
            sort_idx = np.argsort(x)
            x = x[sort_idx]
            y = y[sort_idx]
            y_low = y_low[sort_idx]
            y_high = y_high[sort_idx]

            color = comp_colors.get(comp, 'gray')
            ax.plot(x, y, '-o', label=comp, color=color,
                   markersize=plt_cfg.get('markersize', 5))
            ax.fill_between(x, y_low, y_high, color=color, alpha=0.2)
            ax.axhline(
                np.mean(y), color=color, linestyle='--',
                linewidth=2, alpha=0.7, label=f'Mean {param}'
            )

        if param == 'rs':
            ax.set_yscale('log')
        if param == 'alpha':
            ax.axhline(1, color='black', linestyle='--', linewidth=1.5,
                      label='NFW', alpha=0.8)
        if param == 'beta':
            ax.axhline(3, color='black', linestyle='--', linewidth=1.5,
                      label='NFW', alpha=0.8)

    # Add linear redshift scale on top axis
    def ln1pz_to_z(ln1pz):
        """Convert ln(1+z) to z"""
        return np.exp(ln1pz) - 1

    def z_to_ln1pz(z):
        """Convert z to ln(1+z)"""
        return np.log(1 + z)

    for i, ax in enumerate(axes):
        # create secondary axis
        secax = ax.secondary_xaxis('top', functions=(ln1pz_to_z, z_to_ln1pz))
        
        # y-axis ticks
        ax.tick_params(axis='y', which='both', left=True, right=False,
                       labelleft=True, direction='in')
        # x-axis
        if i == 0:
            ax.legend(loc='best')
            secax.set_xlabel(r'Redshift $z$')
            ax.tick_params(axis='x', which='both', bottom=True, top=False,
                      labelbottom=False, direction='in')
            secax.tick_params(axis='x', which='both', bottom=False, top=True,
                              labeltop=True, direction='in')
        elif i == len(axes) - 1:
            ax.set_xlabel(r'$\ln(1+z)$')
            ax.invert_xaxis()
            ax.tick_params(axis='x', which='both', bottom=True, top=False,
                      labelbottom=True, direction='in')
            secax.tick_params(axis='x', which='both', bottom=False, top=True,
                              labeltop=False, direction='in')
        else:
            ax.tick_params(axis='x', which='both', bottom=True, top=False,
                      labelbottom=False)
            secax.tick_params(axis='x', which='both', bottom=False, top=True,
                              labeltop=False, direction='in')

    fig.suptitle(f"{halo} / {name}\nHalo Shape Parameter Evolution")
    plt.tight_layout()

    if save_fig:
        save_figure(
            fig,
            (f"{PATHS.plots}/{experiment}/{halo}/halo_shape_evolution.png")
            if experiment else f"{PATHS.plots}/{halo}/halo_shape_evolution.png", 
            plt_cfg.get('dpi', 200)
        )
    if show_fig:
        return fig
    else:
        plt.close(fig)



In [ ]:
# plot halo evolution
fig = plot_halo_shape_evolution(
    config=config,
    halo="halo_005036",
    name="Squall",
    PATHS=PATHS,
    save_fig=True,
    show_fig=True,
    experiment="dm_only",
)

In [ ]:
results = load_hdf5(PATHS.results/"dm_only/halo_005036/evolution_time_series.h5")

redshifts = results['time']['redshifts']
print(redshifts)

In [ ]:
data_dir = PATHS.snapshot_data("halo_005036", 0.0, experiment="dmo")
metadata = load_hdf5(data_dir / "metadata.h5")
data = DensityProfile.from_hdf5(data_dir/"3d_profile_dm.h5", component="dm")
print(metadata.get('center_kpc'))
plt.loglog(data.r_1d, data.rho_1d)
plt.axvline(metadata['R200_kpc'], linestyle='--', color='black')
#plt.show()

In [ ]:
%run scripts/diagnose_dmo_profile.py

In [ ]:
%run scripts/check_r200_metadata.py

In [ ]:
import yt

center_full_physics = [40.28397074, 42.48544463, 42.53142123] #Mpc

ds = yt.load("/cephfs/brs/bw_mw/mean/halo_005036/dm-only-L5/DD0064/DD0064")
center = ds.arr(center_full_physics, 'Mpc')
# Whole-box projection (no center needed — defaults to domain center)
proj = yt.ParticleProjectionPlot(
    ds, "z",
    ("all", "particle_mass"),
    width=ds.domain_width[0],  # full box
)
proj.set_unit(("all", "particle_mass"), "Msun")
proj.set_axes_unit("Mpc")
proj.show()

In [ ]:
# Find densest region via mass-weighted 3D histogram
ad = ds.all_data()
positions = np.column_stack([
    ad["all", "particle_position_x"].to("kpc").value,
    ad["all", "particle_position_y"].to("kpc").value,
    ad["all", "particle_position_z"].to("kpc").value,
])
weights = ad["all", "particle_mass"].to("Msun").value

H, edges = np.histogramdd(positions, bins=64, weights=weights)
idx = np.unravel_index(H.argmax(), H.shape)
hist_center = ds.arr(
    [(edges[d][idx[d]] + edges[d][idx[d]+1]) / 2.0 for d in range(3)],
    "kpc"
)
print(f"Histogram center: {hist_center}")
print(f"Histogram center (Mpc): {hist_center.to('Mpc')}")

# Zoom in on it
proj = yt.ParticleProjectionPlot(
    ds, "z",
    ("all", "particle_mass"),
    center=hist_center,
    width=(1000, "kpc"),
    depth=(1000, "kpc"),
)
proj.set_unit(("all", "particle_mass"), "Msun")
proj.set_axes_unit("kpc")
proj.show()


In [ ]:
# Check particle mass distribution to find high-res particle mass
masses = ad["all", "particle_mass"].to("Msun").value
unique_masses, counts = np.unique(np.round(masses, -2), return_counts=True)

for m, c in zip(unique_masses, counts):
    print(f"  Mass ~ {m:.2e} Msun: {c:,} particles")

# The high-res particles will be the lowest mass with the most counts
highres_mass = unique_masses[np.argmax(counts)]
print(f"\nHigh-res particle mass: {highres_mass:.2e} Msun")

# Filter to high-res particles only
mask = np.abs(masses - highres_mass) / highres_mass < 0.1
pos_hr = positions[mask]
print(f"High-res particles: {mask.sum():,} / {len(mask):,}")

# Histogram on high-res particles only (unweighted = particle count)
H, edges = np.histogramdd(pos_hr, bins=128)
idx = np.unravel_index(H.argmax(), H.shape)
hr_center = ds.arr(
    [(edges[d][idx[d]] + edges[d][idx[d]+1]) / 2.0 for d in range(3)],
    "kpc"
)
print(f"High-res center: {hr_center.to('Mpc')}")

# Zoom in
proj = yt.ParticleProjectionPlot(
    ds, "z",
    ("all", "particle_mass"),
    center=hr_center,
    width=(1000, "kpc"),
    depth=(1000, "kpc"),
)
proj.set_unit(("all", "particle_mass"), "Msun")
proj.set_axes_unit("kpc")
proj.show()


# Bug fix: gas_only dm particle filter

**Problem:** `gas_only` sims have no star particles, so yt never creates the `creation_time` field.
The globally registered `"dm"` filter in `filters_and_utils.py` has `requires=["creation_time"]`,
which causes `YTIllDefinedParticleFilter` for every snapshot.

**Fix:** Re-register the `"dm"` filter at runtime using particle mass (same approach as the `dmo` branch),
so boundary particles are excluded without touching `creation_time`.

The cell below tests this on a gas_only snapshot (`DD0001`) — update the path if needed.

In [ ]:
import gc
import yt
import numpy as np
from src.filters_and_utils import filter_highres_particles

snap_path = "/cephfs/brs/bw_mw/mean/halo_005036/gas+dm-L4_redo/DD0001/DD0001"

ds = yt.load(snap_path)

# ── Confirm creation_time is absent ──────────────────────────────────────────
creation_fields = [f for f in ds.field_list if "creation" in str(f)]
print("Fields with 'creation':", creation_fields)

# ── Proposed fix: mass-based dm filter (mirrors the dmo branch) ──────────────
ad_temp = ds.all_data()
all_masses = ad_temp["all", "particle_mass"].to("Msun").value
_, hr_mass_threshold = filter_highres_particles(all_masses)
del ad_temp
gc.collect()

print(f"gas_only: high-res particle mass = {hr_mass_threshold:.2e} Msun")

hr_mass_min = hr_mass_threshold * 0.9
hr_mass_max = hr_mass_threshold * 1.1

def _dm_highres(pfilter, data):
    mass = data["all", "particle_mass"].to("Msun").value
    return (mass >= hr_mass_min) & (mass <= hr_mass_max)

yt.add_particle_filter("dm", function=_dm_highres, filtered_type="all")
ds.add_particle_filter("dm")

# ── Sanity check ─────────────────────────────────────────────────────────────
ad = ds.all_data()
n_dm = len(ad["dm", "particle_mass"])
n_all = len(ad["all", "particle_mass"])
print(f"dm particles (high-res): {n_dm:,}  /  all particles: {n_all:,}")
print(f"fraction selected: {n_dm/n_all:.3f}")


In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

file = h5py.File("datasets/Dump_Growth_Rate_Rotation_eps0_1_test_z1dot3 1.hf5", 'r')

gridsize=200

qvals = file['tabqx'][:,0].reshape(gridsize,gridsize)
xvals = file['tabqx'][:,1].reshape(gridsize,gridsize)
grvals = file['tabGR'][:].reshape(gridsize,gridsize)

qvals,xvals = np.meshgrid(np.linspace(0.,1.0,gridsize),np.concatenate([np.linspace(0.,0.4,gridsize-1), [1.0]]),indexing='ij')

plt.contourf(xvals,qvals,grvals,cmap='inferno_r')
plt.xlabel('p (=m_bulge/(m_bulge+m_disk))')
plt.ylabel('q (=m_disk/(m_disk+m_halo))')
plt.colorbar(label='growth rate (arbitrary)')

plt.show()
 

In [ ]:
import src

fig = src.plot_reddish_2022_figure(
    halo=halos[0],
    name=names[0],
    config=load_config("config/fitting_config_r200.yaml"),
    PATHS=PATHS,
    background_data_file="datasets/Dump_Growth_Rate_Rotation_eps0_1_test_z1dot3 1.hf5",
    show_fig=True,
    experiment="r200_scaling"
)

plt.show()

In [ ]:
import numpy as np

z = np.exp(1.3) - 1
print(z)

In [ ]:
from src import get_plot_config, load_hdf5, ProjectPaths, load_config, save_figure
from typing import Dict, Optional
from pathlib import Path
import h5py

def plot_reddish_2022_figure(
    halo: str,
    name: str,
    config: Dict,
    PATHS: ProjectPaths,
    background_data_file: Optional[Path | str] = None,
    save_fig: bool = False,
    experiment: Optional[str] = None,
    show_fig: bool = False,
    exclude_low_p: float = -2.5,
):
    plt_cfg = get_plot_config(config, 'ratio_evolution')
    base = PATHS.results / experiment if experiment else PATHS.results
    results = load_hdf5(f"{base}/{halo}/evolution_time_series.h5")
    redshifts = results['time']['redshifts']

    masses_gas = results['gas']['mass_evolution']
    masses_dm = results['dm']['mass_evolution']
    masses_star = results['star']['mass_evolution']

    M_bulge = []
    M_disk = []
    M_halo = []
    valid_indices = []  # Track which snapshot indices have data
    snapshot_keys = sorted(k for k in masses_dm if k.startswith("snapshot_"))
    for snapshot in snapshot_keys:
        i = int(snapshot.split("_")[1])

        valid_indices.append(i)
        M_bulge.append(masses_star[snapshot].get('bulge', 0)[-1])
        M_disk.append(
            masses_gas[snapshot].get('disk', 0)[-1] +
            masses_star[snapshot].get('disk', 0)[-1]
        )
        M_halo.append(
            masses_star[snapshot].get('halo', 0)[-1] +
            masses_gas[snapshot].get('halo', 0)[-1] +
            masses_dm[snapshot].get('halo', 0)[-1]
        )

    M_bulge = np.array(M_bulge)
    M_disk = np.array(M_disk)
    M_halo = np.array(M_halo)

    log_p = np.log10(M_bulge / (M_bulge + M_disk))  # Bulge-to-Disk ratio
    log_q = np.log10(M_disk / (M_disk + M_halo))   # Disk-to-Halo ratio

    # Use redshifts only for valid snapshots
    z_orig = np.array([redshifts[i] for i in valid_indices])
    z_plot = np.log(1 + z_orig)

    mask = (log_p > exclude_low_p)
    z_plot, log_p, log_q, z_orig = z_plot[mask], log_p[mask], log_q[mask], z_orig[mask]

    fig, ax = plt.subplots(figsize=plt_cfg.get('figsize', (8, 6)))

    # Plot Reddish+2022 background growth-rate field
    if background_data_file is not None:
        gridsize = 200
        with h5py.File(background_data_file, 'r') as bg:
            grvals = bg['tabGR'][:].reshape(gridsize, gridsize)
        bg_q, bg_p = np.meshgrid(
            np.linspace(0., 1.0, gridsize),
            np.concatenate([np.linspace(0., 0.4, gridsize - 1), [1.0]]),
            indexing='ij'
        )
        # Convert linear p, q to log10 scale to match plot axes
        log_bg_p = np.log10(np.clip(bg_p, bg_p[bg_p > 0].min(), 1))
        log_bg_q = np.log10(np.clip(bg_q, bg_q[bg_q > 0].min(), 1))
        cf = ax.contourf(log_bg_p, log_bg_q, grvals, levels=20,
                         cmap='inferno_r', alpha=0.6, zorder=0)

    # Color by redshift
    cmap = plt.get_cmap(plt_cfg.get('cmap', 'turbo_r'))
    norm = mcolors.Normalize(vmin=min(z_plot), vmax=max(z_plot))

    # Plot points colored by redshift
    sc = ax.scatter(
        log_p, log_q, c=z_plot,
        cmap=cmap, norm=norm,
        s=plt_cfg.get('scatter_size', 60),
        edgecolors='white',
        linewidth=0.5,
        zorder=3
    )

    # Connect points with a line to show evolution track
    ax.plot(
        log_p, log_q, '-',
        color='gray', alpha=0.4,
        linewidth=plt_cfg.get('linewidth', 1),
        zorder=1
    )
    ax.set_xlim(-1.75, 0)
    ax.set_ylim(-1.75, 0)

    # Annotate first and last points (show original z values)
    ax.annotate(f'z={z_orig[0]:.1f}', (log_p[0], log_q[0]),
                xytext=(8, 8), textcoords='offset points', fontsize=9)
    ax.annotate(f'z={z_orig[-1]:.1f}', (log_p[-1], log_q[-1]),
                xytext=(8, -12), textcoords='offset points', fontsize=9)

    # Arrow showing direction of time evolution (midpoint of track)
    if len(log_p) >= 3:
        mid = len(log_p) // 2
        ax.annotate(
            '', xy=(log_p[mid + 1], log_q[mid + 1]),
            xytext=(log_p[mid], log_q[mid]),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
            zorder=2
        )

    # Add colorbar for redshift (right side) — must be created before top colorbar
    # so both are sized against the same final axes width
    cbar = fig.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label(r'$\ln(1+z)$')

    # Add top colorbar for growth rate, now that axes width is finalised
    if background_data_file is not None:
        gr_cbar = fig.colorbar(cf, ax=ax, location='top', pad=0.02, aspect=20)
        gr_cbar.set_label('Growth rate (Reddish+2022)', fontsize=8)
        gr_cbar.ax.tick_params(labelsize=7)

    ax.set_xlabel(r'$\log_{10}(M_{\rm bulge} / (M_{\rm bulge} + M_{\rm disk}))$')
    ax.set_ylabel(r'$\log_{10}(M_{\rm disk} / (M_{\rm disk} + M_{\rm halo}))$')

    # suptitle avoids clashing with the top colorbar
    fig.suptitle(f'{halo} / {name}  —  Mass Ratio Evolution', fontsize=11)

    plt.tight_layout()
    if save_fig:
        if experiment:
            path = PATHS.plots / experiment / halo / "component_mass_ratio_evolution.png"
        else:
            path = PATHS.plots / halo / "component_mass_ratio_evolution.png"
        save_figure(
            fig,
            path,
            plt_cfg.get('dpi', 200)
        )
    if show_fig:
        return fig
    else:
        plt.close(fig)


fig = plot_reddish_2022_figure(
    halo = "halo_008508",
    name="Tempest",
    config=load_config("config/fitting_config_r200.yaml"),
    PATHS=ProjectPaths(
        code_root = Path("/home/shanson/foggie-dynamics"),
        data_root = Path("/cephfs/shanson/FOGGIE")),
    background_data_file="datasets/Dump_Growth_Rate_Rotation_eps0_1_test_z1dot3 1.hf5",
    save_fig=True,
    show_fig=True,
    experiment="r200_scaling"
)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

r = np.logspace(-2, 2, 50, base=10)
rho = lambda alpha, beta, gamma: r**-alpha * (1+r**gamma)**((alpha-beta)/gamma)


fig, ax = plt.subplots(figsize=(7, 6))
ax.loglog(r, rho(1, 3, 1), label="NFW Profile\n" + r"$(\alpha,\beta)=(1,3)$", zorder=10)
#ax.loglog(r, rho(1, 4, 1), label="Hernquist Profile\n" + r"$(\alpha,\beta)=(1,4)$", alpha=0.8)
#ax.loglog(r, rho(2, 4, 1), '--', label=r"$(\alpha,\beta)=(2,4)$", alpha=0.5)
#ax.loglog(r, rho(1, 2, 1), '--', label=r"$(\alpha,\beta)=(1,2)$", alpha=0.5)
ax.loglog(r, rho(1, 3, 2), label=r"$(\alpha,\beta,\gamma)=(1,3,2)$")
ax.loglog(r, rho(1, 3, 0.5), label=r"$(\alpha,\beta,\gamma)=(1,3,0.5)$")
ax.set(
    xlabel = r"Scaled Radius $(r / r_s)$",
    ylabel = r"Scaled Density $(\rho / \rho_0)$",
    title = "Two-Power models",
)
ax.legend()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from src import (
    ProjectPaths,
    get_plot_config,
    save_figure,
    
)

df = pd.read_csv("/Users/samuelhanson/Library/CloudStorage/OneDrive-UniversityofEdinburgh/Year 5/MPhys/foggie-dynamics/results/mass_comparison_detail.csv")

mask = (df['sim_mass'] > 0) & (df['model_mass'] > 0)
df['rel_error'] = np.abs((df['model_mass'] - df['sim_mass'])) / df['sim_mass']
df.loc[~mask, 'rel_error'] = np.nan

component = 'gas'
halo = "halo_005036"
redshifts = np.array(sorted(df['redshift'].unique()))

plot_redshifts = np.log(1 + redshifts)

# Color map: high redshift = lighter, low redshift = darker
cmap = cm.turbo_r
norm = plt.Normalize(vmin=min(plot_redshifts), vmax=np.log(4))

sub = df[
    (df['halo'] == halo) &
    (df['component'] == component) &
    (df['redshift'] <= 3) &
    (df['sim_mass'] > 0) &
    (df['model_mass'] > 0)
]

fig, ax = plt.subplots(figsize=(8, 6))

for z in redshifts:
    if z <= 3:
        sub_z = sub[(sub['redshift'] == z)]
        if sub_z.empty:
            continue
        z_plot = np.log(1+z)
        color = cmap(norm(z_plot))
        ax.plot(sub_z['radius_kpc'], sub_z['rel_error'], color=color, lw=1)

ax.axhline(0.1, linestyle='-', color='grey', alpha=0.4, lw=0.8, label='10%')
ax.axhline(0.5, linestyle='-', color='red', alpha=0.4, lw=0.8, label='50%')
ax.axhline(sub['rel_error'].mean(), linestyle='--', color='black', label='mean')
ax.axhline(sub['rel_error'].median(), linestyle=':', color='black', label='median')
ax.legend()

# Colorbar for redshift
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label='ln(1+z)')

ax.set(
    xscale = "log",
    yscale = "log",
    xlabel = "Radius [kpc]",
    ylabel = "Rel. Residual - (|model - data| / data)"
)

fig.suptitle(f'Mass profiles — {component} - {halo}', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
from src import plot_mass_error, load_config, ProjectPaths
from pathlib import Path
PATHS = ProjectPaths(
    code_root=Path("/Users/samuelhanson/Library/CloudStorage/OneDrive-UniversityofEdinburgh/Year 5/MPhys/foggie-dynamics"),
    data_root=Path("/Users/samuelhanson/Library/CloudStorage/OneDrive-UniversityofEdinburgh/Year 5/MPhys/foggie-dynamics/datasets")
)
config = load_config(PATHS.config_file("fitting_config_r200.yaml"))
halos = ["halo_002392", "halo_002878", "halo_004123", "halo_005016", "halo_005036", "halo_008508"]
names = ['Hurricane', 'Cyclone', 'Blizzard', 'Squall', 'Maelstrom', 'Tempest']

for halo, name in zip(halos, names):
    for comp in ['star', 'gas', 'dm']:
        plot_mass_error(
            "/Users/samuelhanson/Library/CloudStorage/OneDrive-UniversityofEdinburgh/Year 5/MPhys/foggie-dynamics/results/r200_scaling/mass_comparison_detail.csv",
            comp,
            halo,
            name,
            config,
            True,
            False,
            "r200_scaling",
            PATHS,
            vs_redshift=True, vs_redshift_mode='total'
        )


In [ ]:
from datetime import datetime

start = datetime(2026, 3, 9, 22, 52, 50, 349000)
end = datetime(2026, 3, 14, 11, 52, 50, 349000)

print((end - start).total_seconds() / (3600))

In [ ]:
z = 0.05
h = 0.695

l_phys = lambda l_comov, z: l_comov / (1+z)

l_comov = 0.391 / h # kpc comoving

print(l_phys(l_comov, z))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def exp_disc(R, z, rho_0, R_d, z_0):
    return rho_0 * np.exp(-R / R_d) * np.cosh(z / z_0)**-2

def disc_flare(R, z, rho_0, R_d, A, B):
    z_0 = A * R + B
    return exp_disc(R, z, rho_0, R_d, z_0)

r = np.linspace(0.1, 30, 200)
z_arr = np.linspace(0.1, 5, 200)
R, z = np.meshgrid(r, z_arr)

# Model parameters
rho_0, R_d, z_0 = 1.0, 5.0, 1.0
A, B = 0.09, 0.5   # flaring: z_0 increases with R

rho_exp   = exp_disc(R, z, rho_0, R_d, z_0)
rho_flare = disc_flare(R, z, rho_0, R_d, A, B)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, rho, title in zip(
    axes,
    [rho_exp, rho_flare],
    ["Exponential Disc", "Flared Disc"]
):
    im = ax.pcolormesh(R, z, np.log10(rho + 1e-10),
                       cmap="viridis", shading="auto",
                       vmin=-3, vmax=0)
    ax.set_xlabel("R (kpc)", fontsize=13)
    ax.set_title(title, fontsize=14)
    plt.colorbar(im, ax=ax, label=r"$\log_{10}\,\rho/\rho_0$")

axes[0].set_ylabel("z (kpc)", fontsize=13)
fig.tight_layout()
plt.show()

